In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import metpy.calc as mpcalc
import numpy as np
import xarray as xr
import glob
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import pandas as pd
import cmocean.cm as cmo

In [2]:
# get dataset 
lat_min, lat_max = (20,90) #
lon_min, lon_max = (-80,70)


y =np.load("/work/uo1075/u241321/data/y310_T.npy") 

data_x = xr.open_dataset('/work/uo1075/u241321/data/hfx_1970-2019_assi_dt.nc', decode_times=False)  # unit: w
data_y = xr.open_dataset('/work/uo1075/u241321/data/hfy_1970-2019_assi_dt.nc', decode_times=False)

var_x = np.mean(data_x['__xarray_dataarray_variable__'].sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max)), axis=1) 
var_y = np.mean(data_y['__xarray_dataarray_variable__'].sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max)), axis=1) 




In [3]:
# regression onto x and y transport respectively, then calculate manitude with direction

field_x = var_x.stack(spatial=('lat','lon')).dropna(dim="spatial") #time,space

nn = 6 # number of regression year


from sklearn.linear_model import LinearRegression
def regression(x,y):

    coef = LinearRegression(fit_intercept=True).fit(x.reshape(-1, 1), y.values.reshape(-1, 1)).coef_
    

    return coef




coe_x = np.zeros((nn, field_x.shape[1]))

for m in range(0,field_x.shape[1],1):
        coe_x[0,m] = regression(y[7:47], field_x[2:42,m])
        coe_x[1,m] = regression(y[7:47], field_x[3:43,m])
        coe_x[2,m] = regression(y[7:47], field_x[4:44,m])
        coe_x[3,m] = regression(y[7:47], field_x[5:45,m])
        coe_x[4,m] = regression(y[7:47], field_x[6:46,m])
        coe_x[5,m] = regression(y[7:47], field_x[7:47,m])
       


In [4]:
coe_x = xr.DataArray(coe_x,  
                    dims=['mode','spatial'],
                    coords=dict(
                        spatial=field_x.spatial,
                         mode=np.arange(1,nn+1,1))
                    , )
# field = var.stack(spatial=('lat','lon')).dropna(dim="spatial") #time,space
spatial = field_x .coords["spatial"]
mode = coe_x .coords["mode"]
reg_x = xr.DataArray(coe_x, dims = ["mode","spatial"], coords = {"mode":mode,"spatial":spatial}).unstack()  


In [5]:
field_y = var_y.stack(spatial=('lat','lon')).dropna(dim="spatial") #time,space
coe_y = np.zeros((nn, field_y.shape[1]))

for m in range(0,field_y.shape[1],1):
        coe_y[0,m] = regression(y[7:47], field_y[2:42,m])
        coe_y[1,m] = regression(y[7:47], field_y[3:43,m])
        coe_y[2,m] = regression(y[7:47], field_y[4:44,m])
        coe_y[3,m] = regression(y[7:47], field_y[5:45,m])
        coe_y[4,m] = regression(y[7:47], field_y[6:46,m])
        coe_y[5,m] = regression(y[7:47], field_y[7:47,m])


In [6]:
coe_y = xr.DataArray(coe_y,  
                    dims=['mode','spatial'],
                    coords=dict(
                        spatial=field_y.spatial,
                         mode=np.arange(1,nn+1,1))
                    , )

spatial_y = field_y .coords["spatial"]
reg_y = xr.DataArray(coe_y, dims = ["mode","spatial"], coords = {"mode":mode,"spatial":spatial_y}).unstack()        
lon = reg_x.lon
lat = reg_x.lat

In [7]:
x = reg_x.stack(spatial=('mode','lat','lon'))
y = reg_y.stack(spatial=('mode','lat','lon'))
spatial = x.coords["spatial"]

In [8]:
from multiprocessing import Pool
from itertools import product
from itertools import starmap

def mask_insignificant(data1, data2):
    
    mode1 = np.where(np.sqrt(data1*data1+data2*data2) >= 1e+13 , data1, np.NaN )
    mode2 = np.where(np.sqrt(data1*data1+data2*data2) >= 1e+13, data2, np.NaN )
    
    return mode1, mode2


def grid_mask(m):
    
    mode = mask_insignificant(x[m], y[m])
    
    return mode


res = Pool().map(grid_mask,np.arange(0,x.shape[0],1))
re = np.array(res)
# output 
reg_transx = xr.DataArray(re[:,0], dims = ["spatial"], coords = {"spatial":spatial}).unstack()    
reg_transy = xr.DataArray(re[:,1], dims = ["spatial"], coords = {"spatial":spatial}).unstack()    

In [9]:
reg_transx.to_netcdf("/work/uo1075/u241321/data/reg_transx_T_c2_bp_mask.nc")
reg_transy.to_netcdf("/work/uo1075/u241321/data/reg_transy_T_c2_bp_mask.nc")

In [10]:
reg_transx

<xarray.DataArray (mode: 6, lat: 70, lon: 150)>
array([[[-1.58676588e+13, -8.90766018e+12,             nan, ...,
         -6.90958188e+13, -5.89300706e+13, -3.31655258e+13],
        [-2.01092061e+12,             nan, -5.85664659e+12, ...,
         -4.73843469e+13, -2.71479934e+13, -7.97442286e+12],
        [ 2.10315272e+13,             nan,  5.84098205e+12, ...,
         -1.89884354e+13, -9.36616151e+12,             nan],
        ...,
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan],
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan],
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan]],

       [[-3.15747028e+13, -2.43414366e+13, -1.91850229e+13, ...,
          2.64543906e+13,  2.11160016e+12, -7.63584753e+12],
        [-5.31488147e+12, -8.45384885e+12, -2.20986187e+13, ...,
          2.93708585e+13,  1.00921241e+13,             nan],
        [-3.71640657e+12, -1.74298209e+13, -2.54141839e+13, ...,
          9.37776778e+12,             nan,             nan],
...
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan],
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan],
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan]],

       [[ 1.46314696e+13,             nan,  7.07832858e+11, ...,
          1.76687987e+13,             nan,             nan],
        [ 5.25656463e+12,             nan,             nan, ...,
          8.14555325e+12,             nan,             nan],
        [-8.83710816e+11, -1.48912321e+13, -2.05355992e+13, ...,
                     nan,             nan,             nan],
        ...,
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan],
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan],
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan]]])
Coordinates:
  * mode     (mode) int64 1 2 3 4 5 6
  * lat      (lat) float64 20.5 21.5 22.5 23.5 24.5 ... 85.5 86.5 87.5 88.5 89.5
  * lon      (lon) float64 -79.5 -78.5 -77.5 -76.5 -75.5 ... 66.5 67.5 68.5 69.5